In [1]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import torch.nn.functional as F
import math
from collections import deque
GRID_SIZE =15
BATCH_SIZE =64
GAMMA=0.99
LEARNING_RATE =1e-7
MEMORY_SIZE =50000
EPSILON_DECAY =0.99
EPSILON_START =0.05
EPSILON_END=0.1
SAVE_EP=100
TARGET_UPDATE_EP=10
EPISODES=5000
UPDATE_FREQ=4
RESET_EPSILON_EVERY=5000
N_step=1

In [2]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

class SimplePacmanEnv(gym.Env):
    """
    简化的吃豆人环境：
    - 网格大小 grid_size  15
    - 玩家从 (0,0) 开始
    - 随机生成约 60% 的豆子
    - 吃到豆子得 +1，吃完所有豆子得 +10，每步 -0.02
    - 状态: 2通道 (玩家位置, 豆子分布)，每个通道 grid_size x grid_size
    """
    def __init__(self, grid_size=15):
        super().__init__()
        self.grid_size = grid_size

        # 动作空间: 0上,1下,2左,3右
        self.action_space = spaces.Discrete(4)

        # 观察空间: 2通道网格，每个像素值0或1
        self.observation_space = spaces.Box(
            low=0, high=1,
            shape=(2, grid_size, grid_size),
            dtype=np.float32
        )

        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # 玩家起始位置
        self.player_pos = [0, 0]

        # 随机生成豆子（约60%）
        total_cells = self.grid_size * self.grid_size
        num_dots = int(total_cells * 0.6)
        rng = np.random.default_rng(seed)
        indices = rng.choice(total_cells, num_dots, replace=False)
        self.dot_grid = np.zeros((self.grid_size, self.grid_size), dtype=bool)
        self.dot_grid.flat[indices] = True
        self.dot_grid[0, 0] = False  # 玩家起点无豆

        self.dots_left = np.sum(self.dot_grid)
        self.steps = 0
        return self._get_obs(), {}

    def _get_obs(self):
        # 玩家通道
        player = np.zeros((self.grid_size, self.grid_size), dtype=np.float32)
        px, py = self.player_pos
        player[px, py] = 1.0

        # 豆子通道
        dot = self.dot_grid.astype(np.float32)

        return np.stack([player, dot], axis=0)

    def step(self, action):
        self.steps += 1
        reward = -0.05
        done = False

        # 移动玩家
        x, y = self.player_pos
        if action == 0:   # 上
            y = min(y + 1, self.grid_size - 1)
        elif action == 1: # 下
            y = max(y - 1, 0)
        elif action == 2: # 左
            x = max(x - 1, 0)
        elif action == 3: # 右
            x = min(x + 1, self.grid_size - 1)
        self.player_pos = [x, y]

        # 吃豆
        if self.dot_grid[x, y]:
            self.dot_grid[x, y] = False
            self.dots_left -= 1
            reward += 1

        # 检查是否吃完所有豆子
        if self.dots_left == 0:
            reward += 10
            done = True

        # 步数上限
        if self.steps >= 1000:
            done = True

        return self._get_obs(), reward, done, {}

    def render(self, mode='human'):
        # 简单文本渲染
        grid = [['.' for _ in range(self.grid_size)] for _ in range(self.grid_size)]
        # 标记豆子
        for i in range(self.grid_size):
            for j in range(self.grid_size):
                if self.dot_grid[i, j]:
                    grid[i][j] = 'o'
        # 标记玩家
        px, py = self.player_pos
        grid[px][py] = 'P'
        # 打印
        for row in reversed(grid):
            print(' '.join(row))
        print(f"Steps: {self.steps}, Dots left: {self.dots_left}")

In [ ]:
class NoisyLinear(nn.Module):
    def __init__(self,in_features,out_features,std_init=0.5):
        super().__init__()
        self.in_features=in_features#输入特征数
        self.out_features=out_features#输出特征数
        self.weight_mu=nn.Parameter(torch.Tensor(out_features,in_features))#权重均值
        self.bias_mu =nn.Parameter(torch.Tensor(out_features))#偏置均值
        self.weight_sigma=nn.Parameter(torch.Tensor(out_features,in_features))#权重标准差
        self.bias_sigma=nn.Parameter(torch.Tensor(out_features))#偏置标准差
        self.std_init = std_init#标准差初始化值
        self.reset_parameters()
        self.register_buffer("weight_epsilon",torch.Tensor(out_features,in_features))
        self.register_buffer("bias_epsilon",torch.Tensor(out_features))
    def reset_parameters(self):
        mu_range=1/math.sqrt(self.in_features)
        nn.init.uniform_(self.weight_mu, -mu_range, mu_range)
        nn.init.uniform_(self.bias_mu, -mu_range, mu_range)
        nn.init.constant_(self.weight_sigma, self.std_init)
        nn.init.constant_(self.bias_sigma, self.std_init)